# Meta Model — Directional Probability (MFE_up vs MFE_down)

**Goal:** predict P(MFE_long > MFE_short) — the probability that the favorable upward excursion
dominates the favorable downward excursion in the next 8h.

**Training filter:** all bars where `max(mfe_long_pips, mfe_short_pips) >= 70` — i.e. any bar where
a significant move (>= 70 pips) occurred in at least one direction. This is ~10-20x more training
data than filtering on the MFE model's predictions, and includes LONG-dominant bars that the MFE
model may not have flagged (removing the SHORT bias from the previous version).

**Target:** `dir_prob = mfe_long_pips / (mfe_long_pips + mfe_short_pips)` — a value in [0,1].
- Near 1.0 → strongly LONG (up excursion dominates)
- Near 0.0 → strongly SHORT (down excursion dominates)
- Near 0.5 → uncertain (both sides roughly equal)

**Pipeline:**
1. Load all `features_9` parquets (already contain `mfe_long_pips`, `mfe_short_pips`)
2. Filter to `max(mfe_long_pips, mfe_short_pips) >= 70` (raw label filter, no model dependency)
3. Same 308 feature set as MFE model
4. Target: `dir_prob = mfe_long_pips / (mfe_long_pips + mfe_short_pips)`
5. Walk-forward expanding CV — LightGBM binary cross-entropy (treats target as probability)
6. Evaluate: calibration, directional accuracy at various confidence thresholds, per-pair breakdown

**At inference time:** the model is still applied only on MFE signal bars (Q50 >= 70) in the live system.

In [ ]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

import lightgbm as lgb

import warnings
warnings.filterwarnings('ignore')

# Paths
FEATURES_DIR   = Path('../backend/data/features_9')
MFE_MODEL_PATH = Path('../backend/models_9/mfe_q50_8h/model_1H_Q50.joblib')
SAVE_DIR       = Path('../backend/models_9/dir_prob_8h')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Config
TRAIN_END   = '2024-06-30'
MFE_THRESH  = 70.0       # same as live system
N_FOLDS     = 5
TEST_RATIO  = 0.10
CONF_THRESH = 0.65       # predict LONG if dir_prob > CONF_THRESH, SHORT if < (1 - CONF_THRESH)

# Same VOL_DROP as MFE training
VOL_DROP = [
    'atr_24', 'atr_72', 'atr_6', 'atr_ratio_6_24', 'atr_ratio_6_72',
    'vol_regime_5d', 'vol_regime_10d', 'vol_trend',
    'range_width_24', 'range_width_48', 'range_width_5d',
    'rv_zscore_24',
]

print('Config ready.')
print(f'  MFE threshold  : {MFE_THRESH} pips')
print(f'  Train cutoff   : {TRAIN_END}')
print(f'  CV folds       : {N_FOLDS}')
print(f'  Conf threshold : {CONF_THRESH} / {1-CONF_THRESH}')
print(f'  Save dir       : {SAVE_DIR}')

## 1. Load Data

In [ ]:
# Load MFE model — still needed for feature_cols reference
mfe_bundle   = joblib.load(MFE_MODEL_PATH)
feature_cols = mfe_bundle['feature_cols']   # 308 features
print(f'MFE model loaded | {len(feature_cols)} features | {mfe_bundle["n_iters"]} iters')
print(f'(MFE model used for feature_cols only — training filter uses raw labels)')

# Load all features_9 parquets
print('\nLoading features_9 parquets...')
dfs = [pd.read_parquet(f) for f in sorted(FEATURES_DIR.glob('*_features.parquet'))]
df  = pd.concat(dfs).sort_index()
print(f'  {len(df):,} rows | {df.shape[1]} cols | pairs: {df["pair"].nunique()}')
print(f'  mfe_long_pips valid  : {df["mfe_long_pips"].notna().sum():,}')
print(f'  mfe_short_pips valid : {df["mfe_short_pips"].notna().sum():,}')

# Compute directional probability target
total = df['mfe_long_pips'] + df['mfe_short_pips']
df['dir_prob'] = np.where(total > 0, df['mfe_long_pips'] / total, np.nan)

print(f'\nTarget (dir_prob) stats (all data):')
r = df['dir_prob'].dropna()
print(f'  Valid  : {len(r):,}')
print(f'  Mean   : {r.mean():.4f}')
print(f'  % > 0.5 (LONG dominant) : {(r > 0.5).mean():.2%}')
print(f'  % < 0.5 (SHORT dominant): {(r < 0.5).mean():.2%}')

# Train / test split
df_train = df[df.index <= TRAIN_END].copy()
df_test  = df[df.index  > TRAIN_END].copy()
print(f'\nTrain rows : {len(df_train):,}  (<= {TRAIN_END})')
print(f'Test rows  : {len(df_test):,}   (>  {TRAIN_END})')

## 2. Filter Training Set — Raw Label Filter (no model dependency)

In [ ]:
# Filter: keep bars where a >= 70 pip move happened in at least one direction
# This is ~10-20x more data than the MFE model signal filter, and crucially
# includes LONG-dominant bars — removing the SHORT bias of the previous version.
mask_train   = (df_train[['mfe_long_pips','mfe_short_pips']].max(axis=1) >= MFE_THRESH) & \
               df_train['dir_prob'].notna()
df_dir_train = df_train[mask_train].copy()

print(f'Train bars total        : {len(df_train):,}')
print(f'Train bars after filter : {len(df_dir_train):,}  ({len(df_dir_train)/len(df_train):.1%} kept)')

r = df_dir_train['dir_prob']
print(f'\nTarget (dir_prob) on filtered train set:')
print(f'  Mean   : {r.mean():.4f}')
print(f'  Std    : {r.std():.4f}')
print(f'  % > 0.5 (LONG dominant)     : {(r > 0.5).mean():.2%}')
print(f'  % > {CONF_THRESH} (high conf LONG)   : {(r > CONF_THRESH).mean():.2%}')
print(f'  % < {1-CONF_THRESH} (high conf SHORT)  : {(r < 1-CONF_THRESH).mean():.2%}')
print(f'  % uncertain (0.35-0.65)     : {((r >= 1-CONF_THRESH) & (r <= CONF_THRESH)).mean():.2%}')

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
r.hist(bins=50, color='steelblue', edgecolor='white')
plt.axvline(0.5, color='red', linestyle='--', label='0.5 (neutral)')
plt.axvline(CONF_THRESH, color='green', linestyle='--', label=f'{CONF_THRESH} (LONG threshold)')
plt.axvline(1-CONF_THRESH, color='orange', linestyle='--', label=f'{1-CONF_THRESH} (SHORT threshold)')
plt.title('Distribution of dir_prob target (max(mfe_long, mfe_short) >= 70 pip bars)')
plt.xlabel('dir_prob')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Walk-Forward CV — Determine Best Iterations per Quantile

In [ ]:
FIXED_ITERS = 500   # fixed iter count — no early stopping (regime shift causes collapse)

def make_lgbm_params(n_iters=FIXED_ITERS):
    return {
        'objective':         'cross_entropy',
        'metric':            'binary_logloss',
        'boosting_type':     'gbdt',
        'n_estimators':      n_iters,
        'learning_rate':     0.02,
        'num_leaves':        64,
        'max_depth':         6,
        'min_child_samples': 50,
        'feature_fraction':  0.7,
        'bagging_fraction':  0.8,
        'bagging_freq':      5,
        'reg_alpha':         0.1,
        'reg_lambda':        0.1,
        'random_state':      42,
        'n_jobs':            -1,
        'device':            'gpu',
        'verbose':           -1,
    }

# Build expanding-window folds
df_dir_train_sorted = df_dir_train.sort_index().reset_index(drop=False)
n              = len(df_dir_train_sorted)
fold_test_size = int(n * TEST_RATIO)

folds_pos = []
for i in range(N_FOLDS):
    train_end = n - (N_FOLDS - i) * fold_test_size
    test_end  = train_end + fold_test_size
    if train_end < fold_test_size: continue
    folds_pos.append((slice(0, train_end), slice(train_end, test_end)))

print(f'CV folds: {len(folds_pos)}  |  Fixed iters: {FIXED_ITERS}')
idx_col = df_dir_train_sorted.columns[0]
for i, (tr, te) in enumerate(folds_pos):
    tr_dates = df_dir_train_sorted[idx_col].iloc[tr]
    te_dates = df_dir_train_sorted[idx_col].iloc[te]
    print(f'  Fold {i+1}: train {tr_dates.iloc[0].date()} -> {tr_dates.iloc[-1].date()} ({tr.stop:,}) '
          f'| test {te_dates.iloc[0].date()} -> {te_dates.iloc[-1].date()} ({te.stop-te.start:,})')

In [ ]:
# Run CV — fixed iters, no early stopping
X_all = df_dir_train_sorted[feature_cols].ffill().fillna(0).to_numpy()
y_all = df_dir_train_sorted['dir_prob'].to_numpy()

oof_preds  = np.full(n, np.nan)
fold_losses = []

def binary_logloss(y_true, y_pred):
    y_true = np.clip(y_true, 1e-7, 1 - 1e-7)
    y_pred = np.clip(y_pred, 1e-7, 1 - 1e-7)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

print('Running walk-forward CV...\n')
for fold_i, (tr, te) in enumerate(folds_pos):
    X_tr, y_tr = X_all[tr], y_all[tr]
    X_te, y_te = X_all[te], y_all[te]

    params = make_lgbm_params()
    mdl = lgb.LGBMRegressor(**params)
    mdl.fit(X_tr, y_tr, callbacks=[lgb.log_evaluation(period=-1)])

    preds = mdl.predict(X_te)
    oof_preds[te] = preds

    loss      = binary_logloss(y_te, preds)
    fold_losses.append(loss)

    dir_acc   = ((preds > 0.5) == (y_te > 0.5)).mean()
    conf_mask = (preds > CONF_THRESH) | (preds < 1 - CONF_THRESH)
    conf_acc  = ((preds[conf_mask] > 0.5) == (y_te[conf_mask] > 0.5)).mean() if conf_mask.sum() > 0 else np.nan

    print(f'  Fold {fold_i+1}: logloss={loss:.5f}  '
          f'dir_acc={dir_acc:.3f}  conf_acc={conf_acc:.3f} (N_conf={conf_mask.sum():,})')

avg_iters = FIXED_ITERS
avg_loss  = np.mean(fold_losses)

valid     = ~np.isnan(oof_preds)
dir_acc   = ((oof_preds[valid] > 0.5) == (y_all[valid] > 0.5)).mean()
conf_mask = (oof_preds[valid] > CONF_THRESH) | (oof_preds[valid] < 1 - CONF_THRESH)
conf_acc  = ((oof_preds[valid][conf_mask] > 0.5) == (y_all[valid][conf_mask] > 0.5)).mean() if conf_mask.sum() > 0 else np.nan

print(f'\n--- OOF Summary ---')
print(f'  Iters          : {FIXED_ITERS} (fixed)')
print(f'  CV logloss     : {avg_loss:.5f}')
print(f'  OOF dir_acc    : {dir_acc:.3f}  (all bars, null=0.500)')
print(f'  OOF conf_acc   : {conf_acc:.3f}  (|pred-0.5| > {CONF_THRESH-0.5:.2f}, N={conf_mask.sum():,})')

## 4. Train Final Models on Full Training Set

In [ ]:
print(f'Training final model on full training set ({FIXED_ITERS} iters)...')
params = make_lgbm_params(FIXED_ITERS)
params.pop('device', None)

final_model = lgb.LGBMRegressor(**params)
final_model.fit(X_all, y_all, callbacks=[lgb.log_evaluation(period=-1)])

bundle = {
    'model':        final_model,
    'feature_cols': feature_cols,
    'train_end':    TRAIN_END,
    'mfe_thresh':   MFE_THRESH,
    'conf_thresh':  CONF_THRESH,
    'n_iters':      FIXED_ITERS,
    'cv_logloss':   avg_loss,
    'target':       'dir_prob = mfe_long / (mfe_long + mfe_short)',
}
fname = SAVE_DIR / 'model_1H_dir_prob.joblib'
joblib.dump(bundle, fname)
print(f'  Saved: {fname}  ({fname.stat().st_size/1e6:.1f} MB)')
print(f'  Iters: {FIXED_ITERS} | CV logloss: {avg_loss:.5f}')

## 5. OOF Calibration Check

In [ ]:
# OOF Prediction Distribution Analysis
valid = ~np.isnan(oof_preds)
preds_v = oof_preds[valid]
y_v     = y_all[valid]

print('OOF Prediction Distribution:')
print(f'  Mean pred  : {preds_v.mean():.4f}  (true mean: {y_v.mean():.4f})')
print(f'  Std  pred  : {preds_v.std():.4f}  (true std : {y_v.std():.4f})')
print()

# Calibration: bin predictions into deciles, compare avg actual
df_cal = pd.DataFrame({'pred': preds_v, 'actual': y_v})
df_cal['bin'] = pd.qcut(df_cal['pred'], q=10, labels=False)

print(f'  {"Bin":>4} {"Pred range":>20} {"Avg pred":>10} {"Avg actual":>12} {"N":>6}')
print(f'  {"-"*58}')
for b in range(10):
    sub = df_cal[df_cal['bin'] == b]
    print(f'  {b:>4} [{sub["pred"].min():.3f}, {sub["pred"].max():.3f}]  '
          f'{sub["pred"].mean():>10.4f} {sub["actual"].mean():>12.4f} {len(sub):>6,}')

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Calibration plot
grp = df_cal.groupby('bin').agg(avg_pred=('pred','mean'), avg_actual=('actual','mean'))
axes[0].plot([0,1],[0,1],'k--', label='Perfect calibration')
axes[0].scatter(grp['avg_pred'], grp['avg_actual'], color='steelblue', s=80)
axes[0].set_xlabel('Avg predicted dir_prob')
axes[0].set_ylabel('Avg actual dir_prob')
axes[0].set_title('OOF Calibration (by prediction decile)')
axes[0].legend()

# Prediction distribution
axes[1].hist(preds_v, bins=50, color='steelblue', edgecolor='white', alpha=0.7, label='Predictions')
axes[1].hist(y_v, bins=50, color='orange', edgecolor='white', alpha=0.5, label='Actuals')
axes[1].axvline(0.5, color='red', linestyle='--')
axes[1].axvline(CONF_THRESH, color='green', linestyle='--', label=f'Conf thresh {CONF_THRESH}')
axes[1].axvline(1-CONF_THRESH, color='green', linestyle='--')
axes[1].set_title('OOF Prediction vs Actual distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Test Set Evaluation

In [ ]:
# Filter test set — same raw label filter as training (no MFE model dependency)
mask_test   = (df_test[['mfe_long_pips','mfe_short_pips']].max(axis=1) >= MFE_THRESH) & \
              df_test['dir_prob'].notna()
df_dir_test = df_test[mask_test].copy()
print(f'Test bars after filter: {len(df_dir_test):,}  ({len(df_dir_test)/len(df_test):.1%} kept)')
print(f'Test period: {df_dir_test.index.min().date()} -> {df_dir_test.index.max().date()}')

# Also show pair breakdown to check LONG/SHORT balance
print(f'\nPer-pair balance in test set:')
for pair, grp in df_dir_test.groupby('pair'):
    r = grp['dir_prob']
    print(f'  {pair:<10} N={len(grp):>5,}  % LONG dominant: {(r>0.5).mean():.1%}')

X_te  = df_dir_test[feature_cols].ffill().fillna(0).to_numpy()
y_te  = df_dir_test['dir_prob'].values

preds_te = final_model.predict(X_te)
df_dir_test['pred_dir'] = preds_te

# Overall directional accuracy
dir_acc   = ((preds_te > 0.5) == (y_te > 0.5)).mean()
conf_mask = (preds_te > CONF_THRESH) | (preds_te < 1 - CONF_THRESH)
conf_acc  = ((preds_te[conf_mask] > 0.5) == (y_te[conf_mask] > 0.5)).mean() if conf_mask.sum() > 0 else np.nan

print(f'\n{"="*70}')
print(f'  TEST SET RESULTS  (post {TRAIN_END}, max MFE >= {MFE_THRESH})')
print(f'{"="*70}')
print(f'  Dir accuracy (all)            : {dir_acc:.3f}  (null=0.500)')
print(f'  Dir accuracy (|pred-0.5|>0.15): {conf_acc:.3f}  (N={conf_mask.sum():,})')
print(f'  % LONG predicted              : {(preds_te > 0.5).mean():.2%}')
print(f'  % SHORT predicted             : {(preds_te < 0.5).mean():.2%}')
print()

# Accuracy at various confidence thresholds
print(f'  {"Threshold":>12} {"N long":>8} {"N short":>8} {"N total":>8} {"Acc":>8}')
print(f'  {"-"*50}')
for thresh in [0.55, 0.60, 0.65, 0.70, 0.75]:
    mask_l = preds_te > thresh
    mask_s = preds_te < (1 - thresh)
    mask   = mask_l | mask_s
    if mask.sum() < 5:
        continue
    acc = ((preds_te[mask] > 0.5) == (y_te[mask] > 0.5)).mean()
    print(f'  |pred-0.5|>{thresh-0.5:.2f}  {mask_l.sum():>8,} {mask_s.sum():>8,} {mask.sum():>8,} {acc:>8.3f}')

In [ ]:
# Directional accuracy by prediction confidence decile
df_dir_test['conf'] = (df_dir_test['pred_dir'] - 0.5).abs()
df_dir_test['decile'] = pd.qcut(df_dir_test['conf'], q=10, labels=False)
df_dir_test['dir_correct'] = (df_dir_test['pred_dir'] > 0.5) == (df_dir_test['dir_prob'] > 0.5)

print('Dir accuracy by |pred - 0.5| decile (test set):')
print(f'  {"Decile":>7} {"N":>6} {"Avg conf":>10} {"Dir acc":>9}  note')
print(f'  {"-"*50}')
for d in range(10):
    sub = df_dir_test[df_dir_test['decile'] == d]
    if len(sub) < 5: continue
    note = '<-- least confident' if d == 0 else ('<-- most confident' if d == 9 else '')
    print(f'  {d:>7} {len(sub):>6,} {sub["conf"].mean():>10.4f} {sub["dir_correct"].mean():>9.3f}  {note}')

# Per-pair breakdown
print(f'\nPer-pair directional accuracy (test set):')
print(f'  {"Pair":<10} {"N":>6} {"% LONG actual":>15} {"Dir acc":>9} {"Conf acc (>0.15)":>18}')
print(f'  {"-"*65}')
for pair, grp in df_dir_test.groupby('pair'):
    if len(grp) < 10: continue
    pred = grp['pred_dir'].values
    act  = grp['dir_prob'].values
    acc  = ((pred > 0.5) == (act > 0.5)).mean()
    cm   = (pred > CONF_THRESH) | (pred < 1 - CONF_THRESH)
    cacc = ((pred[cm] > 0.5) == (act[cm] > 0.5)).mean() if cm.sum() > 0 else np.nan
    print(f'  {pair:<10} {len(grp):>6,} {(act>0.5).mean():>15.3f} {acc:>9.3f} {cacc:>18.3f} (N={cm.sum():,})')

In [ ]:
# Directional accuracy by hour of day (test set)
df_dir_test['hour'] = df_dir_test.index.hour
print('Dir accuracy by hour (test set):')
print(f'  {"Hour":>5} {"N":>6} {"Dir acc":>9} {"Conf acc":>10}')
print(f'  {"-"*36}')
for h in sorted(df_dir_test['hour'].unique()):
    sub  = df_dir_test[df_dir_test['hour'] == h]
    if len(sub) < 10: continue
    pred = sub['pred_dir'].values
    act  = sub['dir_prob'].values
    acc  = ((pred > 0.5) == (act > 0.5)).mean()
    cm   = (pred > CONF_THRESH) | (pred < 1 - CONF_THRESH)
    cacc = ((pred[cm] > 0.5) == (act[cm] > 0.5)).mean() if cm.sum() > 0 else np.nan
    print(f'  {h:>5} {len(sub):>6,} {acc:>9.3f} {cacc:>10.3f} (N_conf={cm.sum():,})')

## 7. Feature Importance

In [ ]:
import matplotlib.pyplot as plt

imp = pd.Series(final_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle('Feature Importance — Directional Probability Model (dir_prob)', fontsize=14)

# Top 25
imp.head(25)[::-1].plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 25 Features')
axes[0].set_xlabel('Importance (split)')

# Bottom 25 (least useful)
imp.tail(25).plot(kind='barh', ax=axes[1], color='salmon')
axes[1].set_title('Bottom 25 Features (least important)')
axes[1].set_xlabel('Importance (split)')

plt.tight_layout()
plt.savefig(SAVE_DIR / 'feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved feature_importance.png')
print(f'\nTop 10 features:')
for feat, val in imp.head(10).items():
    print(f'  {feat:<40} {val:>6,}')

## 8. Summary

In [ ]:
print('=' * 70)
print('  DIRECTIONAL PROBABILITY MODEL — TRAINING COMPLETE')
print('=' * 70)

fname = SAVE_DIR / 'model_1H_dir_prob.joblib'
print(f'\nModel saved: {fname.resolve()}')
print(f'  Size   : {fname.stat().st_size/1e6:.1f} MB')
print(f'  Iters  : {avg_iters}')
print(f'  Target : dir_prob = mfe_long / (mfe_long + mfe_short)')

print(f'\n-- CV Results (train set, walk-forward) --')
print(f'  Avg logloss : {avg_loss:.5f}')
print(f'  OOF dir_acc : {dir_acc:.4f}  (null=0.500)')

valid_te = ~np.isnan(preds_te)
dir_acc_te = ((preds_te > 0.5) == (y_te > 0.5)).mean()
cm_te = (preds_te > CONF_THRESH) | (preds_te < 1 - CONF_THRESH)
cacc_te = ((preds_te[cm_te] > 0.5) == (y_te[cm_te] > 0.5)).mean() if cm_te.sum() > 0 else np.nan

print(f'\n-- Test Set Results (UNSEEN, post {TRAIN_END}) --')
print(f'  Dir accuracy (all bars)              : {dir_acc_te:.4f}')
print(f'  Dir accuracy (conf >= {CONF_THRESH})           : {cacc_te:.4f}  (N={cm_te.sum():,})')
print(f'  Test bars                            : {len(df_dir_test):,}')
print(f'  Test period                          : {df_dir_test.index.min().date()} -> {df_dir_test.index.max().date()}')

print(f'\n-- Config --')
print(f'  MFE threshold  : {MFE_THRESH} pips')
print(f'  Conf threshold : {CONF_THRESH} (LONG) / {1-CONF_THRESH} (SHORT)')
print(f'  Train bars     : {len(df_dir_train):,}  (out of {len(df_train):,} total train)')
print(f'  Features       : {len(feature_cols)}')
print(f'  Train cutoff   : {TRAIN_END}')

print(f'\n-- Next Steps --')
print(f'  If conf_acc > 0.55 on test set, integrate into test_live_full.py:')
print(f'    pred > {CONF_THRESH}     -> LONG only')
print(f'    pred < {1-CONF_THRESH}     -> SHORT only')
print(f'    {1-CONF_THRESH} <= pred <= {CONF_THRESH} -> dual-side (current behavior)')